In [ ]:
# 1. Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# 2. Install dependencies
!pip install -q open_clip_torch gradio transformers accelerate
!pip install -q einops timm safetensors pillow

In [ ]:
# 3. Imports
import torch
import open_clip
import gradio as gr
from PIL import Image
import requests
from io import BytesIO

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# 4. Load BiomedCLIP
print("Loading BiomedCLIP...")
model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)
model = model.to(device).eval()
print(f"✅ Loaded! VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

In [ ]:
# 5. Configuration
CONFIG = {
    "triage_labels": [
        "Chest X-ray", "Brain MRI", "Abdominal CT", "Histopathology",
        "Ultrasound", "Dermoscopy", "Gross pathology", "Bone X-ray",
        "Lung CT", "Retinal fundus", "Mammography"
    ],
    "gatekeeper_prompts": {
        "positive": "Pathological finding, lesion, tumor, abnormality",
        "negative": "Normal tissue, healthy anatomy, background noise"
    },
    "thresholds": {"triage_confidence": 0.5, "gatekeeper_confidence": 0.6}
}

In [ ]:
# 6. TriMedAgent Class (Lite Version)
class TriMedAgentLite:
    def __init__(self, model, preprocess, tokenizer, config, device):
        self.model = model
        self.preprocess = preprocess
        self.tokenizer = tokenizer
        self.config = config
        self.device = device
        self.current_image = None
        self.triage_result = None
    
    def classify(self, image, labels, template="this is a {}"):
        img_tensor = self.preprocess(image).unsqueeze(0).to(self.device)
        texts = self.tokenizer([template.format(l) for l in labels]).to(self.device)
        
        with torch.no_grad():
            img_feat = self.model.encode_image(img_tensor)
            txt_feat = self.model.encode_text(texts)
            img_feat /= img_feat.norm(dim=-1, keepdim=True)
            txt_feat /= txt_feat.norm(dim=-1, keepdim=True)
            probs = (100.0 * img_feat @ txt_feat.T).softmax(dim=-1).cpu().numpy()[0]
        
        return dict(zip(labels, probs.tolist()))
    
    def triage(self, image):
        preds = self.classify(image, self.config["triage_labels"])
        top = max(preds, key=preds.get)
        self.triage_result = {"modality": top, "confidence": preds[top], "all": preds}
        return self.triage_result
    
    def gatekeeper(self, image, boxes, target="abnormality"):
        if not boxes:
            return []
        w, h = image.size
        threshold = self.config["thresholds"]["gatekeeper_confidence"]
        pos = f"{self.config['gatekeeper_prompts']['positive']} of {target}"
        neg = self.config['gatekeeper_prompts']['negative']
        
        verified = []
        for box in boxes:
            crop = image.crop((int(box[0]*w), int(box[1]*h), int(box[2]*w), int(box[3]*h)))
            preds = self.classify(crop, [pos, neg], "this is {}")
            if preds[pos] >= threshold:
                verified.append(box)
        return verified
    
    def chat(self, message, image=None):
        if image:
            self.current_image = image
            self.triage(image)
        
        if not self.current_image:
            return "Please upload an image first."
        
        t = self.triage_result
        msg = message.lower()
        
        if any(w in msg for w in ["detect", "find", "abnormal"]):
            boxes = [[0.3, 0.3, 0.7, 0.7]]  # Simulated
            verified = self.gatekeeper(self.current_image, boxes)
            return f"🔍 Detection: {len(verified)}/{len(boxes)} verified regions"
        
        if any(w in msg for w in ["what", "type", "classify"]):
            top5 = sorted(t['all'].items(), key=lambda x: x[1], reverse=True)[:5]
            return f"📋 Type: {t['modality']} ({t['confidence']:.1%})\n" + "\n".join([f"  • {l}: {p:.1%}" for l,p in top5])
        
        return f"Analyzing {t['modality']} ({t['confidence']:.1%}). Ask about detection or classification."

agent = TriMedAgentLite(model, preprocess, tokenizer, CONFIG, device)
print("✅ Agent ready!")

In [ ]:
# 7. Test with sample image
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Chest_X-ray_in_influenza_and_Haemophilus_influenzae_-_annotated.jpg/800px-Chest_X-ray_in_influenza_and_Haemophilus_influenzae_-_annotated.jpg"
test_img = Image.open(BytesIO(requests.get(url).content)).convert("RGB")

# Triage test
result = agent.triage(test_img)
print(f"Triage: {result['modality']} ({result['confidence']:.1%})")

# Chat test
print(agent.chat("What type of image is this?", test_img))

In [ ]:
# 8. Gradio UI
def process(image, message, history):
    if image and len(history) == 0:
        agent.triage(image)
        t = agent.triage_result
        welcome = f"Image: {t['modality']} ({t['confidence']:.1%})"
        history.append([None, welcome])
    
    if message:
        response = agent.chat(message, image if len(history) == 0 else None)
        history.append([message, response])
    
    return "", history

with gr.Blocks(title="TriMedAgent") as demo:
    gr.Markdown("# 🏥 TriMedAgent Lite")
    with gr.Row():
        img = gr.Image(type="pil", label="Image")
        chat = gr.Chatbot(label="Chat", height=300)
    msg = gr.Textbox(label="Message")
    msg.submit(process, [img, msg, chat], [msg, chat])

demo.launch(share=True)